In [1]:
import pandas as pd
import numpy as np
import requests

In [2]:
url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"

params = {
    "vs_currency": "usd",
    "days": "90",
    "interval": "daily"
}

response = requests.get(url, params=params)
data = response.json()

In [ ]:
df = pd.DataFrame(data['prices'], columns=['timestamp', 'price'])
print(df.head())    

       timestamp         price
0  1772582400000  68321.617848
1  1772668800000  72669.770165
2  1772755200000  70874.986871
3  1772841600000  68148.283400
4  1772928000000  67271.190778


In [6]:
df['date'] = pd.to_datetime(df['timestamp'], unit='ms')
df = df.drop(columns=['timestamp'])
print(df.head())

          price       date
0  68321.617848 2026-03-04
1  72669.770165 2026-03-05
2  70874.986871 2026-03-06
3  68148.283400 2026-03-07
4  67271.190778 2026-03-08


In [8]:
df['return'] = df['price'].pct_change() * 100
print(df.head())


          price       date    return
0  68321.617848 2026-03-04       NaN
1  72669.770165 2026-03-05  6.364241
2  70874.986871 2026-03-06 -2.469780
3  68148.283400 2026-03-07 -3.847201
4  67271.190778 2026-03-08 -1.287036


In [9]:
mean = df['return'].mean()
std = df['return'].std()

df['z_score'] = (df['return'] - mean) / std

print(df.head(10))

          price       date    return   z_score
0  68321.617848 2026-03-04       NaN       NaN
1  72669.770165 2026-03-05  6.364241  3.157689
2  70874.986871 2026-03-06 -2.469780 -1.279296
3  68148.283400 2026-03-07 -3.847201 -1.971122
4  67271.190778 2026-03-08 -1.287036 -0.685250
5  66036.157824 2026-03-09 -1.835902 -0.960924
6  68459.315371 2026-03-10  3.669441  1.804195
7  69883.008876 2026-03-11  2.079620  1.005690
8  70226.818791 2026-03-12  0.491979  0.208280
9  70544.425355 2026-03-13  0.452258  0.188330


In [10]:
df['is_anomaly'] = df['z_score'].abs() > 2

In [11]:
print(df[df['is_anomaly'] == True])

           price       date    return   z_score  is_anomaly
1   72669.770165 2026-03-05  6.364241  3.157689        True
20  70892.828240 2026-03-24  4.486371  2.214508        True
35  71975.621710 2026-04-08  4.518147  2.230468        True
41  74514.630067 2026-04-14  5.310983  2.628678        True


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['date'],
    y=df['price'],
    mode='lines',
    name='BTC Price',
    line=dict(color='orange')
))

fig.add_trace(go.Scatter(
    x=df[df['is_anomaly'] == True]['date'],
    y=y=df[df['is_anomaly'] == True]['price'],,
    mode='markers',
    name='Anomaly',
    marker=dict(color='red', size=10)
))

fig.show()